In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from climate_attitudes.dataset import Dataset
from climate_attitudes.settings import Config
from climate_attitudes.visualisation import configure_mpl

configure_mpl(Path("../fonts/"))

np.set_printoptions(linewidth=200)

In [ ]:
config = Config(_env_file="../.env")
dataset = Dataset.load(
    config,
    name="reduced_no_imputation",
    with_imputation=False,
    verbose=False,
)

labels = dataset.schema.get_short_names(kind="measurement")

measurements = np.load(
    Path(
        "../reports/thesis/results/data/model/bootstrapped_fit/ising_no_use_covariates_no_structure.npz"
    )
)["params"]
Ys = np.load(
    Path(
        "../reports/thesis/results/data/model/bootstrapped_fit/ising_no_use_covariates_no_structure.npz"
    )
)["Y"]
js = measurements[:, 8:].reshape((-1, 8, 8))
diffs = js - np.swapaxes(js, 1, 2)

# # Isolate the directions with positive means
# mean_diff = diffs.mean(axis=0)
# positive_mean_idxes = np.where(mean_diff > 0)

In [ ]:
labels

In [ ]:
# NOTE: Looks like we have all timesteps in axis 2
final_state = np.load(
    Path(
        "../reports/thesis/results/data/model/all_interventions/ising_05_no_use_covariates.npz"
    )
)["measurements"][:, :, -1, :, :]
final_state_null = np.load(
    Path(
        "../reports/thesis/results/data/model/all_interventions/ising_00_no_use_covariates.npz"
    )
)["measurements"][:, :, -1, :, :]
final_state.shape

In [ ]:
final_state_cc_worry_others = ((final_state[:, :, 3] + 1) / 2).mean(axis=0)
final_state_cc_worry_others_null = ((final_state_null[:, :, 3] + 1) / 2).mean(axis=0)

In [ ]:
fig, axes = plt.subplots(
    figsize=(10, 3), ncols=8, sharey=True, sharex=True, constrained_layout=True
)
for i in range(8):
    sns.histplot(final_state_cc_worry_others[..., i], ax=axes[i])

In [ ]:
print(final_state_cc_worry_others.mean(axis=0))

In [ ]:
import polars as pl

ds = dataset.filter(pl.col("dem_male") == 0).filter_no_nulls()

In [ ]:
ds.indices.collect()

In [ ]:
(
    ds.indices.collect()
    .sort(by=("participant_id", "wave"))
    .select(*ds.schema.get_cols("measurement"))
    .to_numpy()
    .ravel()
    .reshape((-1, 2, 8))
)

In [ ]:
fig, axes = plt.subplots(
    nrows=8, ncols=8, constrained_layout=True, sharex=True, sharey=True
)

for i in range(8):
    for j in range(8):
        if i == j:
            continue
        sns.histplot(diffs[:, i, j], ax=axes[i, j])

In [ ]:
fig, axes = plt.subplots(
    nrows=8, ncols=8, constrained_layout=True, sharex=True, sharey=True
)

for i in range(8):
    for j in range(8):
        if i == j:
            continue
        sns.histplot(js[:, i, j], ax=axes[i, j])

Something skewed and null (in our dataset)

Politics --> Climate worry (others)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp

In [ ]:
samples = np.random.normal(
    0.5, 0.5 / sp.stats.Normal(mu=0, sigma=1).icdf(0.95), size=10_000
)

In [ ]:
sigma = -0.5 / sp.stats.Normal(mu=0, sigma=1).icdf(0.01)

In [ ]:
sigma

In [ ]:
x = np.linspace(-1, 1, 100)
p_map_to_1 = sp.stats.Normal(mu=0, sigma=sigma).cdf(x)

In [ ]:
plt.plot(x, p_map_to_1)

In [ ]:
fig, ax = plt.subplots()
sns.histplot(diffs[:, 5, 3], ax=ax)
ax.axvline(diffs[:, 5, 3].mean())

Something null with a broad range

CC Worry --> CC Worry (others)

In [ ]:
fig, ax = plt.subplots()
sns.histplot(diffs[:, 2, 3], ax=ax)
ax.axvline(diffs[:, 2, 3].mean())

Something with a strong effect in our dataset, and generally a strong effect

Climate policy --> belief in climate change

In [ ]:
fig, ax = plt.subplots()
sns.histplot(diffs[:, 7, 0], ax=ax)
ax.axvline(diffs[:, 7, 0].mean())

Something with a measured effect and smaller spread.

Weather worry --> climate impacts

In [ ]:
fig, ax = plt.subplots()
sns.histplot(diffs[:, 4, 6], ax=ax)
ax.axvline(diffs[:, 4, 6].mean())